# B-Spline Occupancy Forecast Validation

This notebook justifies the B-spline occupancy mechanism used by the MPC and RL-MPC experiments. It forecasts occupancy for every dataset building with the same `FixedBSplineIMCOccupancyPredictor` configuration used by the controllers, then compares those forecasts against the actual `occupant_count` time series.

Two fit scopes are reported:

- `controller_full_series`: fits through the same `fit_bspline_occupancy_model` helper used by the controller pipeline, using the full available occupancy series.
- `january_holdout`: fits the same B-spline IMC model only on January (`time_step < 744`) and tests on February. This is the paper-style no-February-leakage validation.

The evaluation focuses on February, matching the controller evaluation window. A persistence baseline is included: it predicts that the current occupancy state remains unchanged.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path.cwd().resolve()
if not (REPO / 'results').exists() and (REPO.parent / 'results').exists():
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.buildings import DATASET_NAME, dataset_building_entries
from src.occupants.bspline_preferences import FixedBSplineIMCOccupancyPredictor
from src.rlmpc_pipeline import (
    fit_bspline_occupancy_model,
    forecast_occupancy_from_bsplines,
    load_building_occupancy_series,
)

BUILDINGS = dataset_building_entries(DATASET_NAME)
FEB_START = 744
FEB_END = 1415
HORIZONS = [1, 2, 3, 6]
ALPHA_OCC = 0.5
BASE_SEED = 49
FIT_SCOPES = {
    'controller_full_series': 'Controller-fit full series',
    'january_holdout': 'January fit -> February test',
}
PRIMARY_FIT_SCOPE = 'january_holdout'

building_table = pd.DataFrame([
    {'building': b['display'], 'building_id': b['result_id'], 'schema_name': b['name']}
    for b in BUILDINGS
])
display(building_table)

## 2. Forecast And Score

For each building and horizon, the model is given the actual current occupancy state at time `t` and asked to predict occupancy at `t + horizon`. This matches how the controller code uses the fitted B-spline Markov model: it conditions on the current occupancy state and the periodic time-of-day transition model.

The `january_holdout` fit scope trains only on January observations and evaluates only on February observations. The `controller_full_series` fit scope is kept as a pipeline sanity check because it uses exactly the same fitting helper as MPC/RL-MPC.

Metrics:

- `mae_count`, `rmse_count`: error in expected occupant count.
- `accuracy`, `precision`, `recall`, `f1`: binary occupied/unoccupied classification after thresholding with `ALPHA_OCC`.
- `brier`: squared error of the clipped expected occupancy probability against the binary occupied state.
- `persistence_*`: same metrics for the baseline that predicts current occupancy continues.

In [ ]:
def make_bspline_occupancy_model(seed):
    return FixedBSplineIMCOccupancyPredictor(
        period=24,
        spline_degree=3,
        n_internal_knots=4,
        l2_phi=1e-3,
        min_samples_per_state=25,
        maxiter=1000,
        random_state=int(seed),
    )


def fit_occupancy_model_for_scope(building, series, seed, fit_scope):
    if fit_scope == 'controller_full_series':
        model, _ = fit_bspline_occupancy_model(seed=seed, building_name=building['name'])
        return model, int(len(series))

    if fit_scope == 'january_holdout':
        model = make_bspline_occupancy_model(seed)
        model.fit(np.asarray(series[:FEB_START], dtype=int))
        return model, int(min(FEB_START, len(series)))

    raise ValueError(f'Unknown fit scope: {fit_scope}')


def binary_metrics(actual_binary, predicted_binary):
    actual_binary = np.asarray(actual_binary, dtype=bool)
    predicted_binary = np.asarray(predicted_binary, dtype=bool)
    tp = int(np.sum(actual_binary & predicted_binary))
    tn = int(np.sum(~actual_binary & ~predicted_binary))
    fp = int(np.sum(~actual_binary & predicted_binary))
    fn = int(np.sum(actual_binary & ~predicted_binary))
    accuracy = (tp + tn) / max(1, len(actual_binary))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def score_model_forecasts(building, series, model, fit_scope, fit_scope_label, fit_samples):
    rows = []
    forecast_rows = []
    max_t = min(FEB_END, len(series) - 1)

    for horizon in HORIZONS:
        t_values = np.arange(FEB_START, max_t - horizon + 1, dtype=int)
        actual = series[t_values + horizon]
        current = series[t_values]
        predicted = []

        for t, occ_now in zip(t_values, current):
            occ_hat, occ_bin = forecast_occupancy_from_bsplines(
                model,
                current_occupancy=int(occ_now),
                time_step=int(t),
                horizon=int(horizon),
                alpha=ALPHA_OCC,
            )
            predicted.append(float(occ_hat[-1]))

        predicted = np.asarray(predicted, dtype=float)
        predicted_binary = predicted > ALPHA_OCC
        actual_binary = actual > ALPHA_OCC
        persistence = current.astype(float)
        persistence_binary = persistence > ALPHA_OCC

        bs = binary_metrics(actual_binary, predicted_binary)
        ps = binary_metrics(actual_binary, persistence_binary)
        rows.append({
            'fit_scope': fit_scope,
            'fit_scope_label': fit_scope_label,
            'fit_samples': int(fit_samples),
            'test_start': int(FEB_START),
            'test_end': int(max_t),
            'building': building['display'],
            'building_id': building['result_id'],
            'building_name': building['name'],
            'horizon': int(horizon),
            'samples': int(len(t_values)),
            'occupied_rate': float(actual_binary.mean()) if len(actual_binary) else np.nan,
            'max_occupant_count': float(series.max()) if len(series) else np.nan,
            'mae_count': float(np.mean(np.abs(predicted - actual))) if len(actual) else np.nan,
            'rmse_count': float(np.sqrt(np.mean((predicted - actual) ** 2))) if len(actual) else np.nan,
            'brier': float(np.mean((np.clip(predicted, 0.0, 1.0) - actual_binary.astype(float)) ** 2)) if len(actual_binary) else np.nan,
            'persistence_mae_count': float(np.mean(np.abs(persistence - actual))) if len(actual) else np.nan,
            'persistence_accuracy': ps['accuracy'],
            'persistence_f1': ps['f1'],
            **bs,
        })

        if horizon == 1:
            forecast_rows.extend([
                {
                    'fit_scope': fit_scope,
                    'fit_scope_label': fit_scope_label,
                    'building': building['display'],
                    'building_id': building['result_id'],
                    'time_step': int(t),
                    'hour': int(t % 24),
                    'current_occupancy': float(c),
                    'actual_next_occupancy': float(a),
                    'predicted_next_occupancy': float(p),
                    'actual_next_occupied': bool(a > ALPHA_OCC),
                    'predicted_next_occupied': bool(p > ALPHA_OCC),
                }
                for t, c, a, p in zip(t_values, current, actual, predicted)
            ])

    return pd.DataFrame(rows), pd.DataFrame(forecast_rows)


def evaluate_building(building):
    seed = BASE_SEED + 1000 * (int(building['index']) - 1)
    series = np.asarray(load_building_occupancy_series(building['name']), dtype=float)
    metric_frames = []
    forecast_frames = []

    for fit_scope, fit_scope_label in FIT_SCOPES.items():
        model, fit_samples = fit_occupancy_model_for_scope(building, series, seed, fit_scope)
        metrics_i, forecasts_i = score_model_forecasts(
            building,
            series,
            model,
            fit_scope,
            fit_scope_label,
            fit_samples,
        )
        metric_frames.append(metrics_i)
        forecast_frames.append(forecasts_i)

    return pd.concat(metric_frames, ignore_index=True), pd.concat(forecast_frames, ignore_index=True)

## 3. All-Building Results

In [ ]:
metric_frames = []
forecast_frames = []

for building in BUILDINGS:
    print(f"Fitting and scoring {building['display']}: {building['name']}")
    metrics_i, forecasts_i = evaluate_building(building)
    metric_frames.append(metrics_i)
    forecast_frames.append(forecasts_i)

occupancy_metrics = pd.concat(metric_frames, ignore_index=True)
one_step_forecasts = pd.concat(forecast_frames, ignore_index=True)

display(occupancy_metrics)

## 4. Summary Tables

In [ ]:
horizon_summary = occupancy_metrics.groupby(['fit_scope', 'fit_scope_label', 'horizon']).agg(
    buildings=('building_id', 'nunique'),
    samples=('samples', 'sum'),
    fit_samples=('fit_samples', 'mean'),
    occupied_rate=('occupied_rate', 'mean'),
    mae_count=('mae_count', 'mean'),
    rmse_count=('rmse_count', 'mean'),
    accuracy=('accuracy', 'mean'),
    precision=('precision', 'mean'),
    recall=('recall', 'mean'),
    f1=('f1', 'mean'),
    brier=('brier', 'mean'),
    persistence_mae_count=('persistence_mae_count', 'mean'),
    persistence_accuracy=('persistence_accuracy', 'mean'),
    persistence_f1=('persistence_f1', 'mean'),
).sort_index()
display(horizon_summary)

one_step_summary = occupancy_metrics[occupancy_metrics['horizon'] == 1].set_index(
    ['fit_scope', 'fit_scope_label', 'building', 'building_id']
)[
    ['occupied_rate', 'mae_count', 'persistence_mae_count', 'accuracy', 'persistence_accuracy', 'f1', 'persistence_f1', 'brier']
].sort_values(['fit_scope', 'f1'], ascending=[True, False])
display(one_step_summary)

## 5. Aggregate Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for fit_scope, fit_scope_label in FIT_SCOPES.items():
    scope_summary = horizon_summary.loc[(fit_scope, fit_scope_label)]
    axes[0].plot(scope_summary.index, scope_summary['mae_count'], marker='o', label=fit_scope_label)
    axes[1].plot(scope_summary.index, scope_summary['accuracy'], marker='o', label=fit_scope_label)
    axes[2].plot(scope_summary.index, scope_summary['f1'], marker='o', label=fit_scope_label)

persistence_summary = horizon_summary.loc[(PRIMARY_FIT_SCOPE, FIT_SCOPES[PRIMARY_FIT_SCOPE])]
axes[0].plot(persistence_summary.index, persistence_summary['persistence_mae_count'], color='black', marker='o', linewidth=1.5, label='Persistence')
axes[1].plot(persistence_summary.index, persistence_summary['persistence_accuracy'], color='black', marker='o', linewidth=1.5, label='Persistence')
axes[2].plot(persistence_summary.index, persistence_summary['persistence_f1'], color='black', marker='o', linewidth=1.5, label='Persistence')

axes[0].set_title('Count MAE by forecast horizon')
axes[0].set_ylabel('MAE')
axes[1].set_title('Occupied/unoccupied accuracy')
axes[1].set_ylabel('Accuracy')
axes[2].set_title('Occupied-class F1')
axes[2].set_ylabel('F1')

for ax in axes:
    ax.set_xlabel('Forecast horizon [hours]')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()

one_step_plot = occupancy_metrics[
    (occupancy_metrics['horizon'] == 1) & (occupancy_metrics['fit_scope'] == PRIMARY_FIT_SCOPE)
].copy()
one_step_plot['label'] = one_step_plot['building'].str.replace('Building ', 'B', regex=False)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].bar(one_step_plot['label'], one_step_plot['mae_count'], label=FIT_SCOPES[PRIMARY_FIT_SCOPE])
axes[0].plot(one_step_plot['label'], one_step_plot['persistence_mae_count'], color='black', marker='o', linewidth=1.5, label='Persistence')
axes[0].set_ylabel('1-step MAE')
axes[0].set_title('January-fit holdout one-step occupancy-count error by building')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(one_step_plot['label'], one_step_plot['f1'], label=FIT_SCOPES[PRIMARY_FIT_SCOPE])
axes[1].plot(one_step_plot['label'], one_step_plot['persistence_f1'], color='black', marker='o', linewidth=1.5, label='Persistence')
axes[1].set_ylabel('1-step F1')
axes[1].set_title('January-fit holdout one-step occupied-class F1 by building')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

## 6. Per-Building Diagnostics

In [ ]:
for building in BUILDINGS:
    building_id = building['result_id']
    display(Markdown(f"### {building['display']}: `{building['name']}`"))
    metrics_i = occupancy_metrics[occupancy_metrics['building_id'] == building_id].set_index(['fit_scope', 'horizon']).sort_index()
    display(metrics_i[['samples', 'fit_samples', 'occupied_rate', 'mae_count', 'persistence_mae_count', 'accuracy', 'persistence_accuracy', 'f1', 'persistence_f1', 'brier']])

    forecasts_i = one_step_forecasts[
        (one_step_forecasts['building_id'] == building_id) & (one_step_forecasts['fit_scope'] == PRIMARY_FIT_SCOPE)
    ].copy()
    if forecasts_i.empty:
        print('No forecasts available.')
        continue

    preview = forecasts_i.head(7 * 24)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].plot(preview['time_step'], preview['actual_next_occupancy'], label='Actual next occupancy', linewidth=2)
    axes[0].plot(preview['time_step'], preview['predicted_next_occupancy'], label='B-spline expected occupancy', linewidth=2)
    axes[0].step(preview['time_step'], preview['predicted_next_occupied'].astype(float), where='post', label='Predicted occupied', alpha=0.7)
    axes[0].set_title(f"{building['display']} | January-fit holdout | First February week")
    axes[0].set_xlabel('Time step')
    axes[0].set_ylabel('Occupancy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    hourly = forecasts_i.groupby('hour').agg(
        actual_occupied_rate=('actual_next_occupied', 'mean'),
        predicted_occupied_rate=('predicted_next_occupied', 'mean'),
        predicted_expected_occupancy=('predicted_next_occupancy', 'mean'),
    )
    axes[1].plot(hourly.index, hourly['actual_occupied_rate'], marker='o', label='Actual occupied rate')
    axes[1].plot(hourly.index, hourly['predicted_occupied_rate'], marker='o', label='Predicted occupied rate')
    axes[1].plot(hourly.index, hourly['predicted_expected_occupancy'], marker='o', label='Mean expected occupancy')
    axes[1].set_title(f"{building['display']} | January-fit holdout | February hourly profile")
    axes[1].set_xlabel('Hour of day')
    axes[1].set_ylabel('Rate / expected count')
    axes[1].set_xticks(range(0, 24, 3))
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()